# ⚙️ BPS Installation — MAAP Coding Environment ⚙️

"""
Copyright 2025, European Space Agency (ESA) Licensed under ESA Software Community Licence Permissive (Type 3) - v2.4
"""

**Procedure for installing BPS within the MAAP Coding/Experiment environment.**

This notebook guides through the full installation of the BPS Processor Suite (BPS) on the MAAP Coding environment, starting from the BPS bundle tarball up to the verification of all processors.

---

## 📋 Overview

| Step | Description |
|---|---|
| **0. Configuration** | Define BPS version and all relevant paths |
| **1. Extract tarball** | Untar BPS bundle into the correct directory |
| **2. Conda environment** | Create and register the BPS conda environment |
| **3. Install BPS packages** | Build local conda channel and install all BPS processors |
| **4. Patch `set_environment.bash`** | Update environment script with correct paths and add shell alias |
| **5. Verify installation** | Run `--help` on all processors to confirm successful installation |

---

> ⚠️ **Prerequisites**
> - The BPS bundle tarball `bps-bundle-v{ver}.tar.gz` can be downloaded from: https://service.aresys.it/downloads/  
>   Once downloaded, place it in `/home/jovyan/SW/BPS_V{ver}/`
> - `set_environment.bash` must be present in `/home/jovyan/SCRIPTS/CONFIGURATION_FILE/`

## 0. ⚙️ Configuration

Define all the path variables used throughout the notebook.  
Edit `BPS_VERSION` here to target a different BPS release — all paths are derived automatically.

> 📂 **Expected folder structure before starting:**
> ```
> /home/jovyan/SW/BPS_V450/
> └── bps-bundle-v4.5.0.tar.gz
> 
> /home/jovyan/SCRIPTS/CONFIGURATION_FILE/
> └── set_environment.bash
> ```

| Variable | Description |
|---|---|
| `BPS_VERSION` | BPS version to install (e.g. `4.5.0`) |
| `env_DIR` | Conda environments directory (`/home/jovyan/env`) |
| `SW_DIR` | Directory containing the BPS version folders (`/home/jovyan/SW`) |
| `CONDA_ENV` | Conda environment name (e.g. `BPS_450`) |
| `CONFIGURATION_BPS_DIR` | Directory containing `set_environment.bash` and BPS configuration files (`/home/jovyan/SCRIPTS/CONFIGURATION_FILE`) |

In [4]:
HOME           = "/home/jovyan"
BPS_VERSION    = "4.4.3"
env_DIR        = f"{HOME}/env"
SW_DIR         = f"{HOME}/SW"
_ver_compact   = BPS_VERSION.replace(".", "")
CONDA_ENV      = f"BPS_{_ver_compact}"
CONFIGURATION_BPS_DIR = f"{HOME}/SCRIPTS/CONFIGURATION_FILE/BPS_{_ver_compact}"

# Verbose output for processors (true = full logs, false = summary only)
VERBOSE = "false"   # ← change to "true" for full processor logs

print("Configuration:")
print(f"  HOME                  : {HOME}")
print(f"  BPS_VERSION           : {BPS_VERSION}")
print(f"  env_DIR               : {env_DIR}")
print(f"  SW_DIR                : {SW_DIR}")
print(f"  CONDA_ENV             : {CONDA_ENV}")
print(f"  CONFIGURATION_BPS_DIR : {CONFIGURATION_BPS_DIR}")


Configuration:
  HOME                  : /home/jovyan
  BPS_VERSION           : 4.4.3
  env_DIR               : /home/jovyan/env
  SW_DIR                : /home/jovyan/SW
  CONDA_ENV             : BPS_443
  CONFIGURATION_BPS_DIR : /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443


## 1. 📦 Extract tarball

Extracts the BPS bundle tarball from `SW_DIR/BPS_V{ver}/`.

- **`bps-bundle-v*.tar.gz`** → extracted in-place inside `SW_DIR/BPS_V{ver}/` (→ `bundle/`)

> ℹ️ If `bundle/` already exists, extraction is skipped.

In [5]:
%%bash -s "$BPS_VERSION" "$SW_DIR"
BPS_VERSION=$1
SW_DIR=$2

BPS_SW_DIR="${SW_DIR}/BPS_V${BPS_VERSION//./}"
cd "${BPS_SW_DIR}"

# Extract bps-bundle in-place
if [ -d "bundle" ]; then
    echo "ℹ️  bundle/ already extracted, skipping"
else
    tar -xzf "bps-bundle-v${BPS_VERSION}.tar.gz"
    echo "✅ bps-bundle extracted in ${BPS_SW_DIR}"
fi


✅ bps-bundle extracted in /home/jovyan/SW/BPS_V443


## 2. 🐍 Create Conda environment

Creates the conda environment `CONDA_ENV` (e.g. `BPS_450`) with Python 3.12.

- Creates `env_DIR` (`/home/jovyan/env`) if it does not exist
- Registers `env_DIR` as a conda `envs_dirs`
- Skips creation if the environment already exists

In [6]:
%%bash -s "$CONDA_ENV" "$env_DIR" "$VERBOSE"
CONDA_ENV=$1
env_DIR=$2
VERBOSE=$3

# Clean up any previously configured channels to avoid conflicts
conda config --remove-key channels 2>/dev/null || true
conda config --add channels defaults
echo "✅ Channels cleaned up"


# Log file
LOG_FILE="${env_DIR}/processing_env.log"
> "${LOG_FILE}"
echo "📝 Log file: ${LOG_FILE}"

# Create envs_dirs folder if it doesn't exist
if [ ! -d "${env_DIR}" ]; then
    mkdir -p "${env_DIR}"
    echo "✅ Folder ${env_DIR} created"
else
    echo "ℹ️  Folder ${env_DIR} already exists"
fi

# Register envs_dirs in conda config
if [ "${VERBOSE}" = "true" ]; then
    conda config --add envs_dirs "${env_DIR}" 2>&1 | tee -a "${LOG_FILE}"
else
    conda config --add envs_dirs "${env_DIR}" >> "${LOG_FILE}" 2>&1
fi

# Create new environment only if it doesn't exist
if conda env list | grep -qE "(^|[[:space:]])${CONDA_ENV}([[:space:]]|$)"; then
    echo "ℹ️  Environment ${CONDA_ENV} already exists, skipping creation"
else
    echo "  🔄 Creating environment ${CONDA_ENV}..."
    if [ "${VERBOSE}" = "true" ]; then
        conda create --name "${CONDA_ENV}" python=3.12 -y 2>&1 | tee -a "${LOG_FILE}"
    else
        conda create --name "${CONDA_ENV}" python=3.12 -y >> "${LOG_FILE}" 2>&1
    fi
    if [ ${PIPESTATUS[0]} -ne 0 ]; then
        echo "  ❌ Failed to create environment — see ${LOG_FILE}"
        exit 1
    fi
    echo "✅ Environment ${CONDA_ENV} created"
fi

# Final check
echo ""
echo "=== Conda environments ==="
conda env list
echo "📝 Full log: ${LOG_FILE}"

✅ Channels cleaned up
📝 Log file: /home/jovyan/env/processing_env.log
ℹ️  Folder /home/jovyan/env already exists
  🔄 Creating environment BPS_443...
✅ Environment BPS_443 created

=== Conda environments ===

# conda environments:
#
BPS_443                /home/jovyan/env/BPS_443
base                 * /opt/conda

📝 Full log: /home/jovyan/env/processing_env.log


## 3. 📦 Install BPS Packages

Runs the BPS installation inside the conda environment. Equivalent to `install.sh`.

- **Step 1** — installs `conda-build` with `libmamba` solver
- **Step 2** — collects all `.tar.bz2` / `*conda` files and builds a local conda channel (`bps_conda_channel`)
- **Step 3** — registers `fastai`, `conda-forge`, and the local BPS channel
- **Step 4** — installs all BPS processors and dependencies:
  `bps-l1_processor`, `bps-stack_processor`, `bps-l2a_processor`, `bps-l2b_fd_processor`, `bps-l2b_fh_processor`, `bps-l1_framing_processor`, `bps-l2b_agb_processor`, `arepytools`, `arepyextras-quality`

In [7]:
%%bash -s "$CONDA_ENV" "$SW_DIR" "$BPS_VERSION" "$VERBOSE"
CONDA_ENV=$1
SW_DIR=$2
BPS_VERSION=$3
VERBOSE=$4

BUNDLE_DIR="${SW_DIR}/BPS_V${BPS_VERSION//./}/bundle"
cd "${BUNDLE_DIR}"

# Log file
LOG_FILE="${SW_DIR}/processing_installation.log"
> "${LOG_FILE}"
echo "📝 Log file: ${LOG_FILE}"

# ========================================
echo "▶ Step 1: install conda-build"
# ========================================
if [ "${VERBOSE}" = "true" ]; then
    conda run --name "${CONDA_ENV}" conda install -y conda-build --solver=libmamba 2>&1 | tee -a "${LOG_FILE}"
else
    conda run --name "${CONDA_ENV}" conda install -y conda-build --solver=libmamba >> "${LOG_FILE}" 2>&1
fi
if [ ${PIPESTATUS[0]} -ne 0 ]; then echo "❌ Step 1 failed — see ${LOG_FILE}"; exit 1; fi
echo "✅ conda-build installed"

# ========================================
echo ""
echo "▶ Step 2: create BPS conda channel"
# ========================================
BPS_CONDA_CHANNEL="${BUNDLE_DIR}/bps_conda_channel"
mkdir -p "${BPS_CONDA_CHANNEL}/noarch"
find . -type f -name "*.tar.bz2" -exec cp {} "${BPS_CONDA_CHANNEL}/noarch" \;
find . -type f -name "*conda"    -exec cp {} "${BPS_CONDA_CHANNEL}/noarch" \;
if [ "${VERBOSE}" = "true" ]; then
    conda run --name "${CONDA_ENV}" python -m conda_index --verbose "${BPS_CONDA_CHANNEL}" 2>&1 | tee -a "${LOG_FILE}"
else
    conda run --name "${CONDA_ENV}" python -m conda_index --verbose "${BPS_CONDA_CHANNEL}" >> "${LOG_FILE}" 2>&1
fi
if [ ${PIPESTATUS[0]} -ne 0 ]; then echo "❌ Step 2 failed — see ${LOG_FILE}"; exit 1; fi
echo "✅ BPS conda channel created"

# ========================================
echo ""
echo "▶ Step 3: add conda channels"
# ========================================
if [ "${VERBOSE}" = "true" ]; then
    conda run --name "${CONDA_ENV}" conda config --prepend channels fastai 2>&1 | tee -a "${LOG_FILE}"
    conda run --name "${CONDA_ENV}" conda config --prepend channels conda-forge 2>&1 | tee -a "${LOG_FILE}"
    conda run --name "${CONDA_ENV}" conda config --prepend channels "${BPS_CONDA_CHANNEL}" 2>&1 | tee -a "${LOG_FILE}"
else
    conda run --name "${CONDA_ENV}" conda config --prepend channels fastai >> "${LOG_FILE}" 2>&1
    conda run --name "${CONDA_ENV}" conda config --prepend channels conda-forge >> "${LOG_FILE}" 2>&1
    conda run --name "${CONDA_ENV}" conda config --prepend channels "${BPS_CONDA_CHANNEL}" >> "${LOG_FILE}" 2>&1
fi
echo "✅ Channels configured"

# ========================================
echo ""
echo "▶ Step 4: install BPS packages"
# ========================================
echo "  🔄 Installing BPS packages (this may take a while)..."
if [ "${VERBOSE}" = "true" ]; then
    conda run --name "${CONDA_ENV}" conda install -y \
        bps-l1_processor \
        bps-stack_processor \
        bps-l2a_processor \
        bps-l2b_fd_processor \
        bps-l2b_fh_processor \
        bps-l1_framing_processor \
        bps-l2b_agb_processor \
        arepytools \
        arepyextras-quality \
        --solver=libmamba 2>&1 | tee -a "${LOG_FILE}"
else
    conda run --name "${CONDA_ENV}" conda install -y \
        bps-l1_processor \
        bps-stack_processor \
        bps-l2a_processor \
        bps-l2b_fd_processor \
        bps-l2b_fh_processor \
        bps-l1_framing_processor \
        bps-l2b_agb_processor \
        arepytools \
        arepyextras-quality \
        --solver=libmamba >> "${LOG_FILE}" 2>&1
fi
if [ ${PIPESTATUS[0]} -ne 0 ]; then echo "❌ Step 4 failed — see ${LOG_FILE}"; exit 1; fi

echo ""
echo "✅ BPS packages installed"
echo "📝 Full log: ${LOG_FILE}"

📝 Log file: /home/jovyan/SW/processing_installation.log
▶ Step 1: install conda-build
✅ conda-build installed

▶ Step 2: create BPS conda channel
✅ BPS conda channel created

▶ Step 3: add conda channels
✅ Channels configured

▶ Step 4: install BPS packages
  🔄 Installing BPS packages (this may take a while)...

✅ BPS packages installed
📝 Full log: /home/jovyan/SW/processing_installation.log


## 4. 📝 Patch `set_environment.bash`

This step:
1. Takes `set_environment.bash` from `/home/jovyan/SCRIPTS/CONFIGURATION_FILE/`
2. Creates the version-specific folder `CONFIGURATION_BPS_DIR` (e.g. `CONFIGURATION_FILE/BPS_443/`)
3. Copies `set_environment.bash` into it
4. Patches the file with the correct paths for the current installation
5. Adds a shell alias to `~/.bashrc`

- Patches the following variables via `sed`:

| Variable | New value |
|---|---|
| `BPS_PYTHON_ENV` | `CONDA_ENV` (e.g. `BPS_443`) |
| `BPS_TEST_PLAN_PATH` | `SW_DIR/BPS_V{ver}` |
| `BPS_BUNDLE_DIR` | `SW_DIR/BPS_V{ver}/bundle` |
| `BPS_CONDA_CHANNEL` | `SW_DIR/BPS_V{ver}/bundle/bps_conda_channel` |
| `BPS_TDS_PATH` |  `CONFIGURATION_BPS_DIR` (e.g. `CONFIGURATION_FILE/BPS_443`) |

- Adds a shell alias to `~/.bashrc` (e.g. `bps443`) to source `set_environment.bash` quickly

  ```bash
  alias bps443='source /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443/set_environment.bash'
  ```
- Shows a `diff` between the original and the patched file

In [8]:
%%bash -s "$CONDA_ENV" "$SW_DIR" "$CONFIGURATION_BPS_DIR" "$BPS_VERSION"
CONDA_ENV=$1
SW_DIR=$2
CONFIGURATION_BPS_DIR=$3
BPS_VERSION=$4
_ver_compact="${BPS_VERSION//./}"

BASE_CONFIG_DIR="$(dirname ${CONFIGURATION_BPS_DIR})"
SOURCE_SET_ENV="${BASE_CONFIG_DIR}/set_environment.bash"
SET_ENV="${CONFIGURATION_BPS_DIR}/set_environment.bash"
BPS_SW_VERSION_DIR="${SW_DIR}/BPS_V${_ver_compact}"

# 1. Check source set_environment.bash exists in base folder
if [ ! -f "${SOURCE_SET_ENV}" ]; then
    echo "❌ ERROR: ${SOURCE_SET_ENV} not found!"
    echo "   Please place set_environment.bash in ${BASE_CONFIG_DIR}"
    exit 1
fi

# 2. Create version-specific folder
mkdir -p "${CONFIGURATION_BPS_DIR}"
echo "✅ Created: ${CONFIGURATION_BPS_DIR}"

# 3. Copy set_environment.bash into version folder
cp "${SOURCE_SET_ENV}" "${SET_ENV}"
echo "✅ Copied set_environment.bash to ${CONFIGURATION_BPS_DIR}"

# 4. Patch variables
sed -i "s|BPS_PYTHON_ENV=.*|BPS_PYTHON_ENV=${CONDA_ENV}|" "${SET_ENV}"
sed -i "s|BPS_TEST_PLAN_PATH=.*|BPS_TEST_PLAN_PATH=${BPS_SW_VERSION_DIR}|" "${SET_ENV}"
sed -i "s|BPS_BUNDLE_DIR=.*|BPS_BUNDLE_DIR=${BPS_SW_VERSION_DIR}/bundle|" "${SET_ENV}"
sed -i "s|BPS_CONDA_CHANNEL=.*|BPS_CONDA_CHANNEL=${BPS_SW_VERSION_DIR}/bundle/bps_conda_channel|" "${SET_ENV}"
sed -i "s|BPS_TDS_PATH=.*|BPS_TDS_PATH=${CONFIGURATION_BPS_DIR}|" "${SET_ENV}"
echo "✅ set_environment.bash patched"
echo ""
echo "=== diff vs original ==="
diff "${SOURCE_SET_ENV}" "${SET_ENV}" || true

# 5. Add alias to .bashrc (only if not already present)
ALIAS="alias bps${_ver_compact}='source ${SET_ENV}'"
if grep -qF "${ALIAS}" ~/.bashrc; then
    echo "ℹ️  Alias already present in .bashrc, skipping"
else
    echo "${ALIAS}" >> ~/.bashrc
    echo "✅ Alias added to ~/.bashrc: ${ALIAS}"
fi

✅ Created: /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443
✅ Copied set_environment.bash to /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443
✅ set_environment.bash patched

=== diff vs original ===
13c13
< BPS_TDS_PATH=/home/jovyan/BPS/BPS_V443
---
> BPS_TDS_PATH=/home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443
✅ Alias added to ~/.bashrc: alias bps443='source /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443/set_environment.bash'


## 5. ✅ Verify installation

Sources `set_environment.bash` and runs `--help` on all BPS processors to verify the installation.

Processors tested:
- `bps_l1_processor`
- `bps_l1_framing_processor`
- `bps_stack_processor`
- `bps_l2a_processor`
- `bps_l2b_fd_processor`
- `bps_l2b_fh_processor`
- `bps_l2b_agb_processor`

> ✅ = processor found and responding correctly  
> ❌ = processor missing or installation failed

In [9]:
%%bash -s "$CONDA_ENV" "$CONFIGURATION_BPS_DIR" "$BPS_VERSION"
CONDA_ENV=$1
CONFIGURATION_BPS_DIR=$2
BPS_VERSION=$3

_ver_compact="${BPS_VERSION//./}"
SET_ENV="${CONFIGURATION_BPS_DIR}/set_environment.bash"

# Source set_environment.bash
echo "=== Sourcing ${SET_ENV} ==="
source "${SET_ENV}"
echo "✅ Environment activated"
echo ""

# Test all processors
PROCESSORS=(
    "bps_l1_processor"
    "bps_l1_framing_processor"
    "bps_stack_processor"
    "bps_l2a_processor"
    "bps_l2b_fd_processor"
    "bps_l2b_fh_processor"
    "bps_l2b_agb_processor"
)

echo "=== Processor --help checks ==="
for PROC in "${PROCESSORS[@]}"; do
    echo ""
    echo "────────────────────────────────────────"
    echo "--- ${PROC} ---"
    echo "────────────────────────────────────────"
    if conda run --name "${CONDA_ENV}" "${PROC}" --help; then
        echo "✅ ${PROC} OK"
    else
        echo "❌ ${PROC} FAILED"
    fi
done

echo ""
echo "=== Done ==="

=== Sourcing /home/jovyan/SCRIPTS/CONFIGURATION_FILE/BPS_443/set_environment.bash ===
✅ Environment activated

=== Processor --help checks ===

────────────────────────────────────────
--- bps_l1_processor ---
────────────────────────────────────────
Usage: bps_l1_processor [OPTIONS] JOB_ORDER_FILE

  BIOMASS Processing Suite - L1 Processor

  Starts processing from Job Order file

Options:
  --working-dir PATH  Working directory (optional)
  --version           Show processor version and exit
  --help              Show this message and exit.

✅ bps_l1_processor OK

────────────────────────────────────────
--- bps_l1_framing_processor ---
────────────────────────────────────────
Usage: bps_l1_framing_processor [OPTIONS] JOB_ORDER_FILE

  BIOMASS Processing Suite - L1 Framing Processor

  Starts processing from Job Order file

Options:
  --working-dir PATH  Working directory (optional)
  --version           Show processor version and exit
  --help              Show this message and exit